# 02 - Preprocess the classifier training corpus

This notebook normalises the Deceptive Opinion Spam Corpus and the Yelp Fake
Review Dataset, then builds the combined 80/20 stratified training/test corpus
used by the RoBERTa-LoRA classifier.

Key preprocessing decisions from the paper:

* HTML tags, special characters and empty reviews are removed.
* Text shorter than 30 characters is discarded.
* Capitalisation is preserved because all-caps wording is a useful spam cue.
* Each source is split separately so both domains contribute to train and test.
* Duplicate texts are kept so the released training split can be reproduced
  exactly; it contains two duplicated reviews.

## Setup

In [ ]:
import pandas as pd

from spam_detection.config import (
    CLASSIFIER_TEST_PATH,
    CLASSIFIER_TRAIN_PATH,
    OPSPAM_PREPROCESSED_PATH,
    OPSPAM_RAW_PATH,
    YELP_PREPROCESSED_PATH,
    YELP_SAMPLE_RAW_PATH,
)
from spam_detection.preprocessing import (
    build_training_corpus,
    prepare_opspam,
    prepare_yelp_sample,
)
from spam_detection.utils import set_seed

set_seed(42)

## Load raw datasets

In [ ]:
opspam_raw = pd.read_csv(OPSPAM_RAW_PATH)
yelp_raw = pd.read_csv(YELP_SAMPLE_RAW_PATH)

print("OpSpam:", opspam_raw.shape)
print("Yelp:", yelp_raw.shape)
display(opspam_raw.head(3))
display(yelp_raw.head(3))

## Normalise each source

In [ ]:
opspam = prepare_opspam(opspam_raw)
yelp = prepare_yelp_sample(yelp_raw)

opspam.to_csv(OPSPAM_PREPROCESSED_PATH, index=False)
yelp.to_csv(YELP_PREPROCESSED_PATH, index=False)

print("OpSpam labels:")
print(opspam["label"].value_counts())
print("\nYelp labels:")
print(yelp["label"].value_counts())
print("\nEmpty Yelp reviews after cleaning:", int((yelp["text"] == "").sum()))

## Inspect text quality

In [ ]:
opspam_lengths = opspam["text"].str.split().str.len()
yelp_lengths = yelp["text"].str.split().str.len()

summary = pd.DataFrame(
    {
        "OpSpam": opspam_lengths.describe(),
        "Yelp": yelp_lengths.describe(),
    }
)
display(summary)

print("OpSpam duplicate texts:", int(opspam["text"].duplicated().sum()))
print("Yelp duplicate texts:", int(yelp["text"].duplicated().sum()))

## Build the combined 80/20 corpus

`build_training_corpus` splits each source separately with stratification,
concatenates the training and test partitions and filters reviews shorter than
30 characters. An `80/20` split reproduces the released
`new_detector_train.csv` (5,231 rows) and `new_detector_test.csv` (1,309 rows).
The released training split still contains two duplicated review texts, so
duplicates are intentionally not removed.

In [ ]:
train_df, test_df = build_training_corpus(
    opspam_raw,
    yelp_raw,
    test_size=0.2,
    min_chars=30,
    seed=42,
)

train_df.to_csv(CLASSIFIER_TRAIN_PATH, index=False)
test_df.to_csv(CLASSIFIER_TEST_PATH, index=False)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("\nTrain label balance:")
print(train_df["label"].value_counts(normalize=True).round(4))
print("\nTest label balance:")
print(test_df["label"].value_counts(normalize=True).round(4))
print("\nSources in train:")
print(train_df["source"].value_counts())

## Sanity checks

In [ ]:
assert train_df["text"].str.len().min() >= 30
assert test_df["text"].str.len().min() >= 30
assert set(train_df["label"].unique()) == {0, 1}

print("Duplicate texts in train:", int(train_df["text"].duplicated().sum()))

display(train_df.head(3))
display(test_df.head(3))